Question 1: Impact of Preprocessing on Model Performance

A. Compare LinearRegression with and without scaling

In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Age": [22,25,28,30,35,40,45,50,55,60],
    "Experience": [0,2,4,6,10,15,20,25,30,35],
    "Department": ["IT","HR","IT","Sales","HR",
                   "IT","Sales","IT","HR","Sales"],
    "Salary": [25000,28000,32000,35000,42000,
               55000,60000,75000,80000,95000]
})

X = df.drop(columns="Salary")
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

num = ["Age", "Experience"]
cat = ["Department"]

for scale in [False, True]:
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
    ])

    model = Pipeline([
        ("preprocess", pre),
        ("regressor", LinearRegression())
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("\nScaled:", scale)
    print("MAE:", round(mean_absolute_error(y_test, pred), 2))
    print("R2:", round(r2_score(y_test, pred), 4))



Scaled: False
MAE: 2118.09
R2: 0.9832

Scaled: True
MAE: 2118.09
R2: 0.9832


B. Compare KNeighborsRegressor with and without scaling

In [2]:

from sklearn.neighbors import KNeighborsRegressor

for scale in [False, True]:
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
    ])

    model = Pipeline([
        ("preprocess", pre),
        ("regressor", KNeighborsRegressor(n_neighbors=3))
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("\nScaled:", scale)
    print("MAE:", round(mean_absolute_error(y_test, pred), 2))
    print("R2:", round(r2_score(y_test, pred), 4))



Scaled: False
MAE: 5111.11
R2: 0.9221

Scaled: True
MAE: 4555.56
R2: 0.9523


Question 2: ColumnTransformer for Mixed Data

A. ColumnTransformer for Mixed Data

In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.DataFrame({
    "Age":[21,25,30,35,40,45,50,28,33,38],
    "Annual_Income":[20000,25000,30000,35000,45000,50000,60000,28000,40000,48000],
    "City":["Delhi","Mumbai","Bhopal","Delhi","Pune","Mumbai","Bhopal","Pune","Delhi","Mumbai"],
    "Membership":["Basic","Silver","Gold","Basic","Gold","Silver","Gold","Basic","Silver","Gold"],
    "Spending":[2000,3000,4500,4000,6000,6500,8000,3500,5000,7000]
})

num = ["Age","Annual_Income"]
cat = ["City","Membership"]

pre = ColumnTransformer([
    ("scale",StandardScaler(),num),
    ("encode",OneHotEncoder(handle_unknown="ignore"),cat)
])

X = df.drop(columns="Spending")
data = pre.fit_transform(X)

print(data)
print("Output features:", data.shape[1])

[[-1.57951465 -1.50473255  0.          1.          0.          0.
   1.          0.          0.        ]
 [-1.11151031 -1.08906058  0.          0.          1.          0.
   0.          0.          1.        ]
 [-0.52650488 -0.6733886   1.          0.          0.          0.
   0.          1.          0.        ]
 [ 0.05850054 -0.25771663  0.          1.          0.          0.
   1.          0.          0.        ]
 [ 0.64350597  0.57362733  0.          0.          0.          1.
   0.          1.          0.        ]
 [ 1.22851139  0.9892993   0.          0.          1.          0.
   0.          0.          1.        ]
 [ 1.81351682  1.82064325  1.          0.          0.          0.
   0.          1.          0.        ]
 [-0.76050705 -0.83965739  0.          0.          0.          1.
   1.          0.          0.        ]
 [-0.17550163  0.15795535  0.          1.          0.          0.
   0.          0.          1.        ]
 [ 0.4095038   0.82303051  0.          0.          1.  

B. Pipeline, Evaluation and Prediction

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df.drop(columns="Spending")
y = df["Spending"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = Pipeline([
    ("pre", ColumnTransformer([
        ("scale",StandardScaler(),num),
        ("encode",OneHotEncoder(handle_unknown="ignore"),cat)
    ])),
    ("lr",LinearRegression())
])

model.fit(X_train,y_train)
pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test,pred))
print("R2:", r2_score(y_test,pred))

new = pd.DataFrame({
    "Age":[27,36],
    "Annual_Income":[32000,42000],
    "City":["Bhopal","Jaipur"],
    "Membership":["Gold","Silver"]
})

print("Predicted Spending:", model.predict(new))

MAE: 862.5000000000015
R2: 0.5969003378378368
Predicted Spending: [4462.5   5459.375]


Question 3: Preventing Data Leakage with Pipelines

A. Classification Pipeline

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.DataFrame({
    "Age":[25,32,np.nan,45,50,28,38,np.nan,42,35],
    "Income":[30000,45000,50000,np.nan,80000,35000,60000,55000,np.nan,40000],
    "Employment":["Salaried","Business","Salaried","Self-employed",
                  "Business","Salaried",np.nan,"Business","Salaried","Self-employed"],
    "City":["Delhi","Mumbai","Bhopal","Delhi","Pune","Mumbai",
            "Delhi","Pune",np.nan,"Bhopal"],
    "Repaid_Loan":[1,1,0,1,1,0,1,0,1,0]
})

X = df.drop(columns="Repaid_Loan")
y = df["Repaid_Loan"]

num = ["Age","Income"]
cat = ["Employment","City"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

pre = ColumnTransformer([
    ("num", num_pipe, num),
    ("cat", cat_pipe, cat)
])

model = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

Accuracy: 0.6666666666666666
Confusion Matrix:
 [[0 1]
 [0 2]]


B. Difference between Data Leakage and Correct Preprocessing

In [6]:
# Incorrect
pre.fit(X)
X_transformed = pre.transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.3,
    random_state=42, stratify=y
)

Question 4: Machine Learning Pipeline for House Price Prediction

A. Train-Test Split, Pipeline and Evaluation

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Area":[600,750,800,900,1000,1100,1200,1400,
            1600,1800,2000,2200,2400,2600,2800,3000],
    "Bedrooms":[1,1,2,2,2,3,3,3,3,4,4,4,4,5,5,5],
    "Bathrooms":[1,1,1,2,2,2,2,2,3,3,3,3,3,4,4,4],
    "Age":[20,18,15,12,10,9,8,7,6,5,4,4,3,2,2,1],
    "Price":[1500000,1700000,1900000,2200000,
             2400000,2700000,3000000,3300000,
             3600000,4000000,4300000,4700000,
             5000000,5500000,5800000,6200000]
})

X = df.drop(columns="Price")
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

result = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": pred.round(2)
})

print(result)

MAE: 76323.65463215252
R2: 0.9962448693041054
   Actual Price  Predicted Price
0       1500000       1307637.94
1       1700000       1616531.85
2       2700000       2721241.49
3       5800000       5791777.08


B. Five-Fold Cross-Validation

In [8]:
from sklearn.model_selection import cross_val_score

mae = -cross_val_score(
    model, X, y, cv=5, scoring="neg_mean_absolute_error"
)

r2 = cross_val_score(
    model, X, y, cv=5, scoring="r2"
)

print("MAE scores:", mae.round(2))
print("Average MAE:", mae.mean().round(2))

print("R2 scores:", r2.round(3))
print("Average R2:", r2.mean().round(3))

MAE scores: [101009.85  68235.09  46201.58  69502.06  81541.02]
Average MAE: 73297.92
R2 scores: [0.806 0.873 0.961 0.929 0.912]
Average R2: 0.896
